In [1]:
include("CRD_STA.jl")
using Plots
using LinearAlgebra
using NonlinearEigenproblems
using ProgressMeter
using DelimitedFiles
using PyCall

In [48]:
eigval = [-1 ,3]
index = findmin(abs.((eigval)))
index[2]

1

In [129]:
for Tw = 1 : 0.1 : 1.2
        N_cheb = 99
        Mr = 0.3
        gamma = 1.4
        sigma = 0.72
        be_end = 0.2
        be_step = 0.001
        R_end = 100
        R_step = 0.1
        Ro = 1
        Co = 0
        global u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,Co)
        H,T = T_ca(Mr,f,q,w0,gamma,Tw)
        F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
        lam = - (2/3) * T
        kappa = (1/sigma) * T
        global total_i_2 = []
        global total_r_2 = []
    for omega = -0.024
            be_start =  0.08 + (1 - Tw) * 0.1
            R0 = 26 + (1 - Tw) * 20
            global GR = 0
            global tempvec_i_1 = []
            global tempvec_r_1 = []
            global tempvec_i_2 = []
            global tempvec_r_2 = []
            global initial_i= []
            global initial_r= []

            for  R = R0 : 0.25 : 40
                Ma = Mr/R
                global tempvec_i_1 = [0]
                global tempvec_r_1 = [0]
                global tempvec_i_2 = [0]
                global tempvec_r_2 = [0]
                writedlm("output_eig.dat",initial_r_2)
                writedlm("output_GR_$omega _$Tw _$Mr.dat",initial_r_2)
                writedlm("output_GR_data_$omega _$Tw _$Mr.dat",initial_r_2)
                for be = be_start : 0.2 * be_step : be_end
                    A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co)
                    nep = PEP([A0,A1,A2]); 
                    eigval1,eigvec1 = iar(nep,σ = 0.2 , neigs = 1 ,maxit = 500,tol=1e-10)
                    eigval2,eigvec2 = iar(nep,σ = 0.4 , neigs = 1 ,maxit = 500,tol=1e-10)
                    eigval1 = conj(eigval1)
                    eigval2 = conj(eigval2)
                    eigval = [eigval1[1],eigval2[1]]
                    global tempvec_i_1 = [tempvec_i_1;imag(eigval1[1])]
                    global tempvec_r_1 = [tempvec_r_1;real(eigval1[1])]
                    global tempvec_i_2 = [tempvec_i_2;imag(eigval2[1])]
                    global tempvec_r_2 = [tempvec_r_2;real(eigval2[1])]
                    open("output_eig.dat", "a") do io
                        write(io,"R = $R,be = $be,eig = $eigval\n")
                    end
                    if  imag(eigval1[1]) * tempvec_i_1[end - 1,1] < 0 && abs.(imag(eigval1[1])) < 0.03 
                        global initial_i= [omega R be imag(eigval1[1])]
                        global initial_r= [omega R be real(eigval1[1])]
                        global total_r = initial_r
                        global total_i = initial_i
                        global mode = 1
                        break
                    elseif imag(eigval2[1]) * tempvec_i_2[end - 1,1] < 0 && abs.(imag(eigval2[1])) < 0.03
                        global initial_i= [omega R be imag(eigval2[1])]
                        global initial_r= [omega R be real(eigval2[1])]
                        global total_r = initial_r
                        global total_i = initial_i
                        global mode = 2
                        break
                    elseif (length(tempvec_i_1) > 3  && imag(eigval1[1]) > tempvec_i_1[end - 1,1] > tempvec_i_1[end - 2,1] && imag(eigval1[1]) > 0 && tempvec_i_1[end-1,1]>0 && tempvec_i_1[end-2,1] > 0 ) && (length(tempvec_i_2) > 3  && imag(eigval2[1]) > tempvec_i_2[end - 1,1] > tempvec_i_2[end - 2,1]  && imag(eigval2[1]) > 0 && tempvec_i_2[end-1,1] >0 && tempvec_i_2[end-2,1] > 0 )
                        global mode = 0
                        break
                    elseif imag(eigval1[1]) * imag(eigval2[1]) < 0
                        global mode = 0
                        break
                    end
                end
                if  mode == 1 
                    global R_start = initial_i[end,2]
                    global beta = initial_i[end,3]
                    break
                elseif mode == 2
                    global R_start = initial_i[end,2]
                    global beta = initial_i[end,3]
                    break
                end
            end
        for R = R_start  : R_step : R_end
            Ma = Mr/R
            A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,beta,N_cheb,1,0)
            nep = PEP([A0,A1,A2]); 
            eigval_2,eigvec = iar(nep , σ = 0.4 - im * total_i[end,4] , neigs = 2 ,maxit = 500,tol=1e-10)
            eigval_2 = conj(eigval_2)
            index = findmin(imag(eigval_2))
            GR_temp = imag(eigval_2[index[2]]) * R_step
            global GR = GR + abs(GR_temp)
            open("output_GR_$omega _$Tw _$Mr.dat", "a") do io
            write(io,"R = $R , eigmode2 = $eigval_2 , GR = $GR\n")
            end

            open("output_GR_data_$omega _$Tw _$Mr.dat", "a") do io
            write(io,"$beta\t$R\t$GR\n")
            end
            total_i = [omega R beta imag(eigval_2[index[2]])]
            total_r = [omega R beta real(eigval_2[index[2]])]
        end
    end
end

InterruptException: InterruptException:

In [ ]:
for Tw = 0.8 : 0.1 : 1.2
    for omega = -0.1 : -0.01 : -0.15
        original = read("output_GR_data_$omega _$Tw _0.3.dat",String)
        header = "Zone T = \"Tw=$Tw, Ma=0.6, omega=$omega\""
        open("output_GR_data_$omega _$Tw _0.6.dat", "w") do io
            write(io,header * "\n" * original)
        end
    end
end

In [165]:
for omega = -0.024

    for Tw = 1.2
        N_cheb = 99
        Mr = 0.3
        gamma = 1.4
        sigma = 0.72
        be_end = 0.2
        be_step = 0.001
        R_end = 100
        R_step = 0.1
        be_start = 0
        Ro = 1
        Co = 0
        Ma = Mr/R_end
        global u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,Co)
        H,T = T_ca(Mr,f,q,w0,gamma,Tw)
        F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
        lam = - (2/3) * T
        kappa = (1/sigma) * T
        global total_i_2 = []
        global total_r_2 = []
        global GR = 0
        global initial_i= []
        global initial_r= []
        global tempvec_i = []
        global tempvec_r = []
        writedlm("output_eig.dat",initial_r_2)
        writedlm("output_GR_$omega _$Tw _$Mr.dat",initial_r_2)
        writedlm("output_GR_data_$omega _$Tw _$Mr.dat",initial_r_2)
        for be = be_start : 0.2 * be_step : be_end
            A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R_end,Ma,omega,be,N_cheb,Ro,Co)
            nep = PEP([A0,A1,A2]); 
            eigval,eigvec1 = iar(nep,σ = 0.05 , neigs = 1 ,maxit = 500,tol=1e-10)
            eigval = conj(eigval[1])
            global tempvec_i = [tempvec_i;imag(eigval[1])]
            global tempvec_r = [tempvec_r;real(eigval[1])]
            open("output_eig.dat", "a") do io
                write(io,"R = $R_end,be = $be,eig = $eigval,1\n")
            end
            if length(tempvec_i) > 3 && tempvec_i[end - 1, 1] * tempvec_i[end , 1] < 0 
                initial_i = [omega R_end be imag(eigval[1])]
                initial_r = [omega R_end be real(eigval[1])]
                global beta = initial_i[end,3]
                global guess = initial_r[end,4] - im * initial_i[end,4]
                global tempvec_i = []
                global tempvec_r = []
                break
            end
        end
        for be = beta : 0.2 * be_step : be_end
            A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R_end,Ma,omega,be,N_cheb,Ro,Co)
            nep = PEP([A0,A1,A2]); 
            eigval,eigvec1 = iar(nep,σ = guess , neigs = 1 ,maxit = 500,tol=1e-10)
            eigval = conj(eigval[1])
            global tempvec_i = [tempvec_i;imag(eigval[1])]
            global tempvec_r = [tempvec_r;real(eigval[1])]
            if  length(tempvec_i) > 10 && tempvec_r[end - 1 ,1] > tempvec_r[end ,1] && tempvec_i[end, 1] < 0
                global type1max = tempvec_i[end - 1,1]
                global be1 = be
                global tempvec_i = [tempvec_i;-0.01]
                global tempvec_r = [tempvec_r;0.5]
                global guess = tempvec_r[end ,1] - im * tempvec_i[end, 1]
                A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R_end,Ma,omega,be,N_cheb,Ro,Co)
                nep = PEP([A0,A1,A2]); 
                eigval1,eigvec1 = iar(nep,σ = guess , neigs = 1 ,maxit = 500,tol=1e-10)
                eigval1 = conj(eigval[1])
                A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R_end,Ma ,omega ,be + 0.2 * be_step ,N_cheb ,Ro ,Co)
                nep = PEP([A0,A1,A2]); 
                eigval2,eigvec1 = iar(nep,σ = guess , neigs = 1 ,maxit = 500,tol=1e-10)
                eigval2 = conj(eigval[1])
                if imag(eigval2[1]) > imag(eigval1[1]) 
                    global dir = - 1
                    global guess = eigval1
                    global beta = be
                    global tempvec_i = []
                    global tempvec_r = []
                else
                    global dir = 1
                    global guess = eigval2
                    global beta = be + 0.2 * be_step
                    global tempvec_i = []
                    global tempvec_r = []
                end
                break
            else
                global guess = tempvec_r[end ,1] - im * tempvec_i[end, 1]
            end
            open("output_eig.dat", "a") do io
                write(io,"R = $R_end,be = $be,eig = $eigval,2\n")
            end
        end
        for be = beta : dir * be_step : be_end
            A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R_end,Ma,omega,be,N_cheb,Ro,Co)
            nep = PEP([A0,A1,A2]); 
            eigval,eigvec1 = iar(nep,σ = guess , neigs = 1 ,maxit = 500,tol=1e-10)
            eigval = conj(eigval[1])
            global tempvec_i = [tempvec_i;imag(eigval[1])]
            global tempvec_r = [tempvec_r;real(eigval[1])]
            if length(tempvec_i) > 3 && tempvec_i[end - 1 ,1] > tempvec_i[end ,1]
                global type2max = tempvec_i[end - 1,1]
                global be2 = be
            end
            open("output_eig.dat", "a") do io
            write(io,"R = $R_end,be = $be,eig = $eigval,3\n")
            end
        end
        if type1max > type2max
            global beta = be1
        else
            global beta = be2
        end
    end
end
    #         for R = R_start  : R_step : R_end
    #             Ma = Mr/R
    #             A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,beta,N_cheb,1,0)
    #             nep = PEP([A0,A1,A2]); 
    #             eigval_2,eigvec = iar(nep , σ = 0.4 - im * total_i[end,4] , neigs = 2 ,maxit = 500,tol=1e-10)
    #             eigval_2 = conj(eigval_2)
    #             index = findmin(imag(eigval_2))
    #             GR_temp = imag(eigval_2[index[2]]) * R_step
    #             global GR = GR + abs(GR_temp)
    #             open("output_GR_$omega _$Tw _$Mr.dat", "a") do io
    #             write(io,"R = $R , eigmode2 = $eigval_2 , GR = $GR\n")
    #             end

    #             open("output_GR_data_$omega _$Tw _$Mr.dat", "a") do io
    #             write(io,"$beta\t$R\t$GR\n")
    #             end
    #             total_i = [omega R beta imag(eigval_2[index[2]])]
    #             total_r = [omega R beta real(eigval_2[index[2]])]
    #         end
    #     end
    # end
# There is some problme between the shift of cycle 2 and cycle 3.

InterruptException: InterruptException: